In [ ]:
import numpy as np  # Importe la bibliothèque numpy pour les calculs matriciels

class Transformer:  # Définit une classe pour un Transformer
    def __init__(self, input_dim, d_model, n_heads, output_dim):
        # Initialise les composants du Transformer
        self.self_attention = SelfAttention(input_dim, d_model, n_heads)  # Mécanisme d'attention
        self.fc = np.random.randn(d_model, output_dim)  # Couche finale (fully connected)
        self.d_model = d_model  # Dimension des embeddings

    def forward(self, x):
        # x : (seq_len, input_dim) → propagation avant
        x = self.self_attention.forward(x)  # Applique l'attention (seq_len, d_model)
        x = x @ self.fc  # Couche finale (seq_len, output_dim)
        return x

    def predict(self, sequence):
        # Prédiction sur une séquence d'entrées
        outputs = []  # Liste pour stocker les sorties
        for x in sequence:
            x_onehot = np.zeros((1, len(self.self_attention.WQ)))  # Vecteur one-hot (1, input_dim)
            x_onehot[0, x] = 1  # Met à 1 l'index correspondant à x
            output = self.forward(x_onehot)  # (1, output_dim)
            outputs.append(output)
        return outputs

class SelfAttention:  # Définit une classe pour l'attention multi-tête
    def __init__(self, input_dim, d_model, n_heads):
        self.n_heads = n_heads  # Nombre de têtes d'attention
        self.d_head = d_model // n_heads  # Dimension par tête
        self.WQ = np.random.randn(input_dim, d_model)  # Poids Query
        self.WK = np.random.randn(input_dim, d_model)  # Poids Key
        self.WV = np.random.randn(input_dim, d_model)  # Poids Value
        self.WO = np.random.randn(d_model, d_model)  # Poids de sortie
        self.d_model = d_model  # Dimension des embeddings

    def scaled_dot_product_attention(self, Q, K, V):
        # Calcul de l'attention (Scaled Dot-Product)
        scores = Q @ K.T / np.sqrt(self.d_head)  # Produit scalaire + scaling
        weights = np.softmax(scores, axis=1)  # Poids d'attention
        return weights @ V  # Somme pondérée des V

    def split_heads(self, x):
        # x : (seq_len, d_model) → (n_heads, seq_len, d_head)
        return x.reshape(-1, self.n_heads, self.d_head).transpose(1, 0, 2)

    def forward(self, x):
        # x : (seq_len, input_dim) → calcul de Q, K, V
        Q = self.split_heads(x @ self.WQ)  # (n_heads, seq_len, d_head)
        K = self.split_heads(x @ self.WK)
        V = self.split_heads(x @ self.WV)
        outputs = []
        for i in range(self.n_heads):
            output = self.scaled_dot_product_attention(Q[i], K[i], V[i])
            outputs.append(output)
        outputs = np.concatenate(outputs, axis=1)  # (seq_len, d_model)
        return outputs @ self.WO  # Sortie finale

# Exemple
vocab_size = 5  # Taille du vocabulaire
d_model = 4     # Dimension des embeddings
n_heads = 2      # Nombre de têtes d'attention
output_dim = 3   # Dimension de sortie
transformer = Transformer(vocab_size, d_model, n_heads, output_dim)

sequence = [1, 2]  # Séquence d'entrée
outputs = transformer.predict(sequence)
print(outputs)  # Affiche les sorties
